# Logic-of-Thought: minimal demonstration

This notebook implements the core **logic extension** stage from Liu et al. (NAACL 2025), then translates the newly deduced implications into simple English and appends them to the original context. It uses only the Python standard library, so it runs directly in Google Colab.

Paper: https://aclanthology.org/2025.naacl-long.510/

## 1. A tiny structured extraction

The paper uses an LLM to extract propositions and implications. For a free, deterministic demo, we provide that small intermediate representation directly. Literals use `not ` for negation.

In [ ]:
example = {
    "context": [
        "If you have no keyboarding skills at all, you will not be able to use a computer.",
        "If you are not able to use a computer, you will not be able to write essays using a word processor.",
    ],
    "propositions": {
        "A": "you have keyboarding skills",
        "B": "you are able to use a computer",
        "C": "you are able to write essays using a word processor",
    },
    "implications": [
        {"if": "not A", "then": "not B"},
        {"if": "not B", "then": "not C"},
    ],
}
example

## 2. Logic extension

The paper uses three laws: double negation, contraposition `(p → q) ⇔ (¬q → ¬p)`, and transitivity `(p → q) ∧ (q → r) ⇒ (p → r)`. Here double negation is normalized by representing a literal as `(symbol, is_negated)`.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, order=True)
class Literal:
    symbol: str
    negated: bool = False

    def negate(self):
        return Literal(self.symbol, not self.negated)

    def __str__(self):
        return ("¬" if self.negated else "") + self.symbol

@dataclass(frozen=True)
class Implication:
    antecedent: Literal
    consequent: Literal

    def contraposition(self):
        return Implication(self.consequent.negate(), self.antecedent.negate())

    def __str__(self):
        return f"{self.antecedent} → {self.consequent}"

def parse_literal(text):
    text = text.strip()
    return Literal(text[4:], True) if text.startswith("not ") else Literal(text)

def parse_implication(item):
    return Implication(parse_literal(item["if"]), parse_literal(item["then"]))

def extend_logic(seed_rules):
    """Close a finite set of literal implications under contraposition and transitivity."""
    rules = set(seed_rules)
    changed = True
    while changed:
        changed = False
        additions = {rule.contraposition() for rule in rules}
        for first in rules:
            for second in rules:
                if first.consequent == second.antecedent:
                    additions.add(Implication(first.antecedent, second.consequent))
        additions -= rules
        if additions:
            rules.update(additions)
            changed = True
    return rules

seed_rules = {parse_implication(item) for item in example["implications"]}
all_rules = extend_logic(seed_rules)
new_rules = all_rules - seed_rules

print("Extracted:")
for rule in sorted(seed_rules, key=str): print("  ", rule)
print("\nNewly available after extension:")
for rule in sorted(new_rules, key=str): print("  ", rule)

## 3. Translation and context augmentation

The full method uses an LLM for translation. This template keeps the demo local. We select `C → A`, the deduction highlighted in the paper's worked example.

In [ ]:
def literal_to_text(literal, propositions):
    phrase = propositions[literal.symbol]
    return f"not ({phrase})" if literal.negated else phrase

def implication_to_text(rule, propositions):
    left = literal_to_text(rule.antecedent, propositions)
    right = literal_to_text(rule.consequent, propositions)
    return f"If {left}, then {right}."

target = Implication(Literal("C"), Literal("A"))
assert target in all_rules
augmentation = implication_to_text(target, example["propositions"])
extended_context = example["context"] + [augmentation]

print("Deduction:", target)
print("Translation:", augmentation)
print("\nExtended context:")
for i, sentence in enumerate(extended_context, 1):
    print(f"{i}. {sentence}")

## Try your own example

Edit the `example` dictionary and rerun the notebook. Give each atomic proposition a symbol, express negation as `not A`, and supply direct implications only. The extension function will derive all implications reachable with the paper's contraposition and transitivity laws.

This compact version is for understanding the mechanism—not benchmark reproduction. Real text needs a reliable extraction step, and the paper reports that extraction errors were its largest source of mistakes.